In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy import stats
from datetime import datetime

print("All libraries loaded successfully")

All libraries loaded successfully


In [2]:
# Portfolio: 5 assets across different sectors
# Weights reflect a typical balanced portfolio
tickers = ['SPY', 'TLT', 'GLD', 'XLE', 'EEM']
weights = np.array([0.40, 0.25, 0.15, 0.10, 0.10])  # must sum to 1

# Pull 15 years — need enough history to cover 2008 crisis
start = '2008-01-01'
end = datetime.today().strftime('%Y-%m-%d')

print(f"Downloading data from {start} to {end}...")
raw = yf.download(tickers, start=start, end=end, auto_adjust=True)
prices = raw['Close'].dropna()

returns = prices.pct_change().dropna()

print(f"\nShape: {prices.shape}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nWeights: {dict(zip(tickers, weights))}")
print(f"Weight sum: {weights.sum()}")

# Portfolio daily returns
portfolio_returns = (returns * weights).sum(axis=1)
print(f"\nPortfolio daily return stats:")
print(f"  Mean:  {portfolio_returns.mean()*100:.3f}%")
print(f"  Stdev: {portfolio_returns.std()*100:.3f}%")

[*********************100%***********************]  5 of 5 completed


Shape: (4622, 5)
Date range: 2008-01-02 to 2026-05-15

Weights: {'SPY': 0.4, 'TLT': 0.25, 'GLD': 0.15, 'XLE': 0.1, 'EEM': 0.1}
Weight sum: 1.0

Portfolio daily return stats:
  Mean:  0.035%
  Stdev: 1.063%


In [3]:
confidence = 0.95
alpha = 1 - confidence

# METHOD 1: Historical Simulation
# Sort actual returns and find the 5th percentile
hist_var = np.percentile(portfolio_returns, alpha * 100)
hist_cvar = portfolio_returns[portfolio_returns <= hist_var].mean()

# METHOD 2: Parametric (Normal Distribution)
mu = portfolio_returns.mean()
sigma = portfolio_returns.std()
param_var = stats.norm.ppf(alpha, mu, sigma)
param_cvar = mu - sigma * stats.norm.pdf(stats.norm.ppf(alpha)) / alpha

# METHOD 3: Monte Carlo Simulation
np.random.seed(42)
n_simulations = 100000
mc_returns = np.random.normal(mu, sigma, n_simulations)
mc_var = np.percentile(mc_returns, alpha * 100)
mc_cvar = mc_returns[mc_returns <= mc_var].mean()

print("=" * 55)
print("   VALUE AT RISK — THREE METHODS (95% Confidence)")
print("=" * 55)
print(f"{'Method':<25} {'VaR':>10} {'CVaR':>10}")
print("-" * 55)
print(f"{'Historical Simulation':<25} {hist_var*100:>9.3f}% {hist_cvar*100:>9.3f}%")
print(f"{'Parametric (Normal)':<25} {param_var*100:>9.3f}% {param_cvar*100:>9.3f}%")
print(f"{'Monte Carlo':<25} {mc_var*100:>9.3f}% {mc_cvar*100:>9.3f}%")
print("=" * 55)
print(f"\nInterpretation: On any given day, there is a 5% chance")
print(f"the portfolio loses more than ~{abs(hist_var)*100:.2f}% (historical VaR)")

   VALUE AT RISK — THREE METHODS (95% Confidence)
Method                           VaR       CVaR
-------------------------------------------------------
Historical Simulation        -1.470%    -2.465%
Parametric (Normal)          -1.714%    -2.158%
Monte Carlo                  -1.712%    -2.160%

Interpretation: On any given day, there is a 5% chance
the portfolio loses more than ~1.47% (historical VaR)


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

methods = [
    ('Historical Simulation', portfolio_returns, hist_var, hist_cvar, '#2196F3'),
    ('Parametric (Normal)', portfolio_returns, param_var, param_cvar, '#4CAF50'),
    ('Monte Carlo', pd.Series(mc_returns), mc_var, mc_cvar, '#FF9800'),
]

for ax, (title, data, var, cvar, color) in zip(axes, methods):
    ax.hist(data, bins=100, alpha=0.6, color=color, density=True)
    ax.axvline(var, color='red', linestyle='--', linewidth=1.5, label=f'VaR: {var*100:.2f}%')
    ax.axvline(cvar, color='darkred', linestyle='--', linewidth=1.5, label=f'CVaR: {cvar*100:.2f}%')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('VaR & CVaR — Three Methods Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('var_comparison.png', dpi=150, bbox_inches='tight')
print("Saved: var_comparison.png")

Saved: var_comparison.png


In [5]:
# Define crisis periods
scenarios = {
    '2008 Financial Crisis': ('2008-09-01', '2009-03-31'),
    'COVID Crash':           ('2020-02-19', '2020-03-23'),
    '2022 Rate Hikes':       ('2022-01-01', '2022-12-31'),
}

print("=" * 65)
print("   STRESS TEST — PORTFOLIO PERFORMANCE IN CRISIS PERIODS")
print("=" * 65)
print(f"{'Scenario':<25} {'Period':<25} {'Return':>8} {'Max DD':>8} {'VaR(95)':>8}")
print("-" * 65)

scenario_results = {}

for name, (start_d, end_d) in scenarios.items():
    # Filter returns to crisis period
    mask = (portfolio_returns.index >= start_d) & (portfolio_returns.index <= end_d)
    crisis_returns = portfolio_returns[mask]

    if len(crisis_returns) == 0:
        continue

    # Cumulative return during crisis
    cum_ret = (1 + crisis_returns).prod() - 1

    # Max drawdown during crisis
    cum_curve = (1 + crisis_returns).cumprod()
    rolling_max = cum_curve.cummax()
    drawdown = (cum_curve - rolling_max) / rolling_max
    max_dd = drawdown.min()

    # VaR during crisis
    crisis_var = np.percentile(crisis_returns, 5)

    scenario_results[name] = {
        'returns': crisis_returns,
        'cum_ret': cum_ret,
        'max_dd': max_dd,
        'var': crisis_var
    }

    period_str = f"{start_d} to {end_d[:7]}"
    print(f"{name:<25} {period_str:<25} {cum_ret*100:>7.1f}% {max_dd*100:>7.1f}% {crisis_var*100:>7.2f}%")

print("=" * 65)
print(f"\nBaseline (full period) VaR: {hist_var*100:.2f}%")
print(f"Note: Crisis VaR >> baseline VaR — models trained on calm")
print(f"periods systematically underestimate tail risk in crises.")

   STRESS TEST — PORTFOLIO PERFORMANCE IN CRISIS PERIODS
Scenario                  Period                      Return   Max DD  VaR(95)
-----------------------------------------------------------------
2008 Financial Crisis     2008-09-01 to 2009-03       -18.8%   -34.4%   -5.09%
COVID Crash               2020-02-19 to 2020-03       -23.5%   -24.9%   -7.49%
2022 Rate Hikes           2022-01-01 to 2022-12        -9.4%   -19.7%   -1.58%

Baseline (full period) VaR: -1.47%
Note: Crisis VaR >> baseline VaR — models trained on calm
periods systematically underestimate tail risk in crises.


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['#F44336', '#FF9800', '#9C27B0']

for ax, (name, data), color in zip(axes, scenario_results.items(), colors):
    crisis_returns = data['returns']
    cum_curve = (1 + crisis_returns).cumprod()

    ax.plot(cum_curve.index, cum_curve.values, color=color, linewidth=1.5)
    ax.fill_between(cum_curve.index, cum_curve.values, 1,
                    where=(cum_curve.values < 1),
                    alpha=0.3, color=color)
    ax.axhline(y=1, color='gray', linestyle=':', linewidth=0.8)
    ax.set_title(f"{name}\nReturn: {data['cum_ret']*100:.1f}% | Max DD: {data['max_dd']*100:.1f}%",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Portfolio Value ($1 invested)')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Portfolio Stress Test — Crisis Period Performance',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stress_test.png', dpi=150, bbox_inches='tight')
print("Saved: stress_test.png")

Saved: stress_test.png


In [8]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardise returns before PCA
scaler = StandardScaler()
returns_scaled = scaler.fit_transform(returns)

# Fit PCA
pca = PCA()
pca.fit(returns_scaled)

# Explained variance
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

print("=" * 50)
print("   PCA FACTOR DECOMPOSITION")
print("=" * 50)
print(f"{'Factor':<10} {'Explained Var':>15} {'Cumulative':>12}")
print("-" * 50)
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var)):
    print(f"{'PC' + str(i+1):<10} {ev*100:>14.1f}% {cv*100:>11.1f}%")
print("=" * 50)

# Factor loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=tickers,
    columns=[f'PC{i+1}' for i in range(len(tickers))]
)

print("\nFactor Loadings (how each asset contributes to each factor):")
print(loadings.round(3).to_string())

print(f"\nPC1 explains {explained_var[0]*100:.1f}% of portfolio variance")
print(f"PC1+PC2 explain {cumulative_var[1]*100:.1f}% of portfolio variance")
print(f"\nInterpretation: PC1 is likely the broad market factor —")
print(f"high loadings on SPY, EEM, XLE and negative on TLT (flight to safety)")

   PCA FACTOR DECOMPOSITION
Factor       Explained Var   Cumulative
--------------------------------------------------
PC1                  53.2%        53.2%
PC2                  23.1%        76.3%
PC3                  13.7%        90.0%
PC4                   6.8%        96.8%
PC5                   3.2%       100.0%

Factor Loadings (how each asset contributes to each factor):
       PC1    PC2    PC3    PC4    PC5
SPY  0.553  0.128  0.172 -0.491 -0.638
TLT  0.077  0.812 -0.567 -0.037  0.105
GLD  0.564  0.013  0.250 -0.246  0.747
XLE -0.292  0.567  0.763  0.102 -0.011
EEM  0.534  0.046  0.057  0.829 -0.150

PC1 explains 53.2% of portfolio variance
PC1+PC2 explain 76.3% of portfolio variance

Interpretation: PC1 is likely the broad market factor —
high loadings on SPY, EEM, XLE and negative on TLT (flight to safety)


In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Scree plot
ax1.bar(range(1, len(explained_var)+1), explained_var*100, 
        color='#2196F3', alpha=0.7, label='Individual')
ax1.plot(range(1, len(cumulative_var)+1), cumulative_var*100, 
         'ro-', linewidth=2, label='Cumulative')
ax1.axhline(y=90, color='gray', linestyle='--', linewidth=0.8, label='90% threshold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('Scree Plot — Variance Explained by Each Factor', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Factor loadings heatmap
sns.heatmap(loadings[['PC1', 'PC2', 'PC3']], 
            annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax2)
ax2.set_title('Factor Loadings — First 3 Principal Components', fontweight='bold')
ax2.set_xlabel('Factor')
ax2.set_ylabel('Asset')

plt.tight_layout()
plt.savefig('pca_decomposition.png', dpi=150, bbox_inches='tight')
print("Saved: pca_decomposition.png")

Saved: pca_decomposition.png
